In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random


In [3]:
def load_pairs_from_txt(file_path):
    pairs = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                hindi, english = line.rstrip("\n").split("\t")
                pairs.append((hindi, english))
    return pairs

In [12]:
pairs_data = load_pairs_from_txt("hindi_english.txt")
print(len(pairs_data))
print(pairs_data[0])

10000
('आदमी रात को दफ़्तर में फूल बेचता है ।', 'The man sells flowers in the office at night.')


In [16]:
def split_data_test_train(pairs, train_frac=0.8, val_frac=0.1, seed=42):
    pairs_copy = pairs.copy()
    pairs_data = pairs[:]
    random.Random(seed).shuffle(pairs_data)

    pairs_len = len(pairs_data)
    train_size = int(pairs_len * train_frac)
    val_size = int(pairs_len * val_frac)

    train_pairs = pairs_data[:train_size]
    validation_pairs = pairs_data[train_size:train_size + val_size]
    test_pairs = pairs_data[train_size + val_size:]

    return train_pairs, validation_pairs, test_pairs

In [18]:
train_pairs, val_pairs, test_pairs = split_data_test_train(pairs_data)
print("Train pairs: " , len(train_pairs))
print("Validation pairs: ", len(val_pairs))
print("Test pairs:", len(test_pairs))

Train pairs:  8000
Validation pairs:  1000
Test pairs: 1000


In [19]:
def tokenize_text_line(sentence):
    # Tokenize the sentence into words
    tokens = sentence.split()
    return tokens

In [ ]:
def build_vocab_from_sentences(sentences):
    special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
    all_tokens = sorted({ s_tok for sentence in sentences for s_tok in tokenize_text_line(sentence) })
    itos = special_tokens + all_tokens
    stoi = {}
    for i, tok in enumerate(itos):
        stoi[tok] = i
    return stoi, itos

In [ ]:
def encode_sentence(sentence, stoi):
    tokens = tokenize_text_line(sentence)
    encoded = []
    for token in tokens:
        if token in stoi:
            encoded.append(stoi[token])
        else:
            encoded.append(stoi['<UNK>'])
    encoded = [stoi['<SOS>']] + encoded + [stoi['<EOS>']]
    return encoded

In [ ]:
def decode_sequence(encoded_ids, itos):
    skip = ['<PAD>', '<SOS>', '<EOS>']
    words = []
    for i in encoded_ids:
        if i < len(itos) and itos[i] not in skip:
            words.append(itos[i])
    return ' '.join(words)